In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
import os

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"
ds_janelia = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file://{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

x = ds_janelia.read().result()

In [ ]:
x.shape

In [ ]:
fig, axs = plt.subplots(50, figsize=(10, 20), dpi=200)
for i in range(0, 50):
  ax = axs[i]
  ax.plot(x[2500:3000, i*20],'k',linewidth=1)
  format_ax(ax)
  # ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# from scipy.signal import welch

# fs = 1.0
signal = x[:, :10_000].ravel()
f, Pxx = welch(signal, fs=fs, nperseg=256)

# np.random.seed(0)
# # white_noise = np.random.normal(loc=0, scale=1, size=(1_000_000,))
# f_noise, Pxx_noise = welch(white_noise, fs=fs, nperseg=256)

Pxx_norm = Pxx / np.trapezoid(Pxx, f)
# Pxx_noise_norm = Pxx_noise / np.trapezoid(Pxx_noise, f_noise)

fig, ax = plt.subplots(figsize=(5, 2), dpi=500)
ax.semilogy(f, Pxx_norm, color='k', label='$\\alpha$')
ax.semilogy(f_noise, Pxx_noise_norm, color='r', linestyle='--', label='$W_t$')
ax.set_xlabel('$f$', labelpad=-5, fontsize=12)
ax.set_xscale('log')
ax.legend()
# format_ax(ax)
plt.tight_layout()
plt.show()


In [ ]:
x.shape

mat files

In [ ]:
data_struct['stimset'][0, 0]

In [ ]:
print('freq: --', data_struct['fpsec'].ravel()[0])
print('dur:  --', data_struct['timelists'][0, 0].ravel().max() / (1.97 * 60))

In [ ]:
conditions = [0, 1, 2, 3]
condition = 3
ix_by_condition = np.where(data_struct['stim_full'].ravel() == condition)[0]

In [ ]:
interval_ix = np.where(np.diff(ix_by_condition) != 1)[0]
condition_intervals = tuple()
condition_intervals_by_condition = tuple()
for i, j in zip(np.insert(interval_ix, 0, -1), np.insert(interval_ix, len(interval_ix), len(ix_by_condition)-1)):
  condition_intervals_by_condition = condition_intervals + ((int(ix_by_condition[i+1]), int(ix_by_condition[j])),)

In [ ]:
condition_intervals